In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import joblib
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder


In [26]:
# Read data named preprocessed_data
data = pd.read_pickle('data/data_output/preprocessed_data.pkl')

In [27]:
data.columns

Index(['id', 'item_id', 'store_id', 'd', 'sold', 'date', 'wday', 'month',
       'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sold_lag_1', 'sold_lag_2',
       'sold_lag_3', 'sold_lag_6', 'sold_lag_12', 'sold_lag_24', 'sold_lag_36',
       'iteam_sold_avg', 'state_sold_avg', 'store_sold_avg', 'cat_sold_avg',
       'dept_sold_avg', 'cat_dept_sold_avg', 'store_item_sold_avg',
       'cat_item_sold_avg', 'dept_item_sold_avg', 'state_store_sold_avg',
       'state_store_cat_sold_avg', 'store_cat_dept_sold_avg',
       'rolling_sold_mean', 'expanding_sold_mean'],
      dtype='object')

In [28]:
data.head()

,id,item_id,store_id,d,sold,date,wday,month,year,event_name_1,...,dept_sold_avg,cat_dept_sold_avg,store_item_sold_avg,cat_item_sold_avg,dept_item_sold_avg,state_store_sold_avg,state_store_cat_sold_avg,store_cat_dept_sold_avg,rolling_sold_mean,expanding_sold_mean
1097640,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,CA_1,37,0,2011-03-06,2,3,2011,no_event,...,0.705566,0.705566,0.326172,0.219604,0.219604,1.323242,0.813477,1.035156,0.000000,0.000000
1097641,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,CA_1,37,0,2011-03-06,2,3,2011,no_event,...,0.705566,0.705566,0.257568,0.263428,0.263428,1.323242,0.813477,1.035156,0.000000,0.000000
1097642,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,CA_1,37,0,2011-03-06,2,3,2011,no_event,...,0.705566,0.705566,0.159180,0.077820,0.077820,1.323242,0.813477,1.035156,0.000000,0.000000
1097643,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,CA_1,37,2,2011-03-06,2,3,2011,no_event,...,0.705566,0.705566,1.718750,2.041016,2.041016,1.323242,0.813477,1.035156,0.285645,0.054047
1097644,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,CA_1,37,0,2011-03-06,2,3,2011,no_event,...,0.705566,0.705566,0.972656,0.766602,0.766602,1.323242,0.813477,1.035156,0.000000,0.000000


In [29]:
CA_1 = data[data['store_id'] == 'CA_1']

In [56]:
!mkdir

# for item in tqdm(CA_1.item_id.unique()):
#     one_item = CA_1[CA_1.item_id == item][["sold", "date", "d","month", "wday",'event_name_1',
#        'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX',
#        'snap_WI', 'sold_lag_1', 'sold_lag_2', 'sold_lag_3', 'sold_lag_6',
#        'sold_lag_12', 'sold_lag_24', 'sold_lag_36', 'iteam_sold_avg',
#        'state_sold_avg', 'store_sold_avg', 'cat_sold_avg', 'dept_sold_avg',
#        'cat_dept_sold_avg', 'store_item_sold_avg', 'cat_item_sold_avg',
#        'dept_item_sold_avg', 'state_store_sold_avg',
#        'state_store_cat_sold_avg', 'store_cat_dept_sold_avg',
#        'rolling_sold_mean', 'expanding_sold_mean']]
    
#     joblib.dump(one_item.values, f"./data/data_output/id/{item}.npy")


for item in tqdm(CA_1.item_id.unique()):
    one_item = CA_1[CA_1.item_id == item][["sold", "d","month", "wday", 'snap_CA', 'snap_TX',
       'snap_WI', 'sold_lag_1', 'sold_lag_2', 'sold_lag_3', 'sold_lag_6',
       'sold_lag_12', 'sold_lag_24', 'sold_lag_36', 'iteam_sold_avg',
       'state_sold_avg', 'store_sold_avg', 'cat_sold_avg', 'dept_sold_avg',
       'cat_dept_sold_avg', 'store_item_sold_avg', 'cat_item_sold_avg',
       'dept_item_sold_avg', 'state_store_sold_avg',
       'state_store_cat_sold_avg', 'store_cat_dept_sold_avg',
       'rolling_sold_mean', 'expanding_sold_mean']]
    
    joblib.dump(one_item.values, f"./data/data_output/id/{item}.npy")

usage: mkdir [-pv] [-m mode] directory_name ...


100%|██████████| 3049/3049 [07:26<00:00,  6.83it/s]


In [32]:
item_npy = joblib.load(f"data/data_output/id/HOBBIES_1_001.npy")

In [45]:
item_npy.shape

(1905, 33)

In [36]:
# create dataframe for loading npy files and  train valid split

data_info = CA_1[["item_id", "d"]]

# total number of days -> 1913
# for training we are taking data between 1800 < train <- 1913-28-28 = 1857

train_df = data_info[(1800 < data_info.d) &( data_info.d< 1857)]

# valid data is given last day -> 1885 we need to predict next 28days

valid_df = data_info[data_info.d == 1885]

In [41]:
train_df.head()

,item_id,d
54882000,HOBBIES_1_001,1801
54882001,HOBBIES_1_002,1801
54882002,HOBBIES_1_003,1801
54882003,HOBBIES_1_004,1801
54882004,HOBBIES_1_005,1801


In [43]:
label = LabelEncoder()
label.fit(train_df.item_id)
label.transform(["FOODS_3_827"])

array([1436])

In [53]:
class DataLoading:
    def __init__(self, df, train_window = 28, predicting_window=28):
        self.df = df.values
        self.train_window = train_window
        self.predicting_window = predicting_window

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, item):
        df_item = self.df[item]
        item_id = df_item[0]
        d = df_item[1]
        
        item_npy = joblib.load(f"./data/data_output/id/{item_id}.npy")
        item_npy_demand = item_npy[:,0]
        features = item_npy[d-self.train_window:d]
        print(features)
    

        predicted_demand = item_npy_demand[d:d+self.predicting_window]

        item_label = label.transform([item_id])
        item_onehot = [0] * 3049
        item_onehot[item_label[0]] = 1

        list_features = []
        for f in features:
            one_f = []
            one_f.extend(item_onehot)
            one_f.extend(f)
            list_features.append(one_f)

        return {
            "features" : torch.Tensor(list_features),
            "label" : torch.Tensor(predicted_demand)
        }

In [54]:
datac = DataLoading(train_df)
n = datac.__getitem__(100)
n["features"].shape, n["label"].shape

[[0 '2016-01-12' 1810 1 4 'no_event' 'no_event' 'no_event' 'no_event' 0 1
  1 0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.64794921875 1.2333984375 1.3232421875
  0.56884765625 0.70556640625 0.70556640625 0.8330078125 0.64794921875
  0.64794921875 1.3232421875 0.8134765625 1.03515625 0.428466796875
  0.8701171875]
 [0 '2016-01-13' 1811 1 5 'no_event' 'no_event' 'no_event' 'no_event' 0 1
  0 0.0 0.0 1.0 1.0 0.0 0.0 0.0 0.64794921875 1.2333984375 1.3232421875
  0.56884765625 0.70556640625 0.70556640625 0.8330078125 0.64794921875
  0.64794921875 1.3232421875 0.8134765625 1.03515625 0.428466796875
  0.86962890625]
 [0 '2016-01-14' 1812 1 6 'no_event' 'no_event' 'no_event' 'no_event' 0 0
  1 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.64794921875 1.2333984375 1.3232421875
  0.56884765625 0.70556640625 0.70556640625 0.8330078125 0.64794921875
  0.64794921875 1.3232421875 0.8134765625 1.03515625 0.28564453125
  0.869140625]
 [0 '2016-01-15' 1813 1 7 'no_event' 'no_event' 'no_event' 'no_event' 0 1
  1 0.0 0.0 0.0 0.0 0.0 

TypeError: must be real number, not str